This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from pint import Quantity

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


## Experiment ID Input

In [ ]:
# isotope_exps = [247, 251, 257, 258]
isotope_exps = [15, 16, 17]
isotope_exps = [f"TB-{id}" for id in isotope_exps]
# bg_exp = "ID-245"
# experiment_ids = [*isotope_exps, bg_exp]
experiment_ids = [*isotope_exps]
# isotope_name_lookup = {id: name for name, id in zip(isotopes, isotope_exps)}

In [ ]:
experiment_ids

In [ ]:
calib_input = helpers.get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
detector_code = helpers.get_input_required(
    """\
Which detector was used?
1: Original detector (detector 1)
2: New detector (detector 2)
""",
    [Detector.ZERO, Detector.ONE],
    lambda x: Detector(int(x)-1)
)

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
fit_input = helpers.get_input_with_default(
    """\
Which bimodal fit type do you want to use?
1: Bounds based
2: Peak finder based (default)
Press Enter for default
""",
    default_fit_input,
    int
)

fit_styles: dict[int, SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
done = False
strategy_factory = NeutronStrategyFactory()

while not done:
    window_input = helpers.get_input_with_default(
        """\
Which neutron classification window do you want to use?
1: NASA window (default)
2: Neutron distribution window
Press Enter for default
""",
        1,
        int
    )
    load_window_input = helpers.get_input_with_default(
        """\
Do you want to load the borders from the standard border file?
[y/n, or press Enter for no]
""",
        "n",
        str
    )
    done = True
    will_load = load_window_input == "y"

    try:
        if window_input == 1:
            if will_load:
                settings = get_nasa_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", True, settings
                )
            else:
                settings = get_nasa_generation_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", False, settings
                )
                pass
        elif window_input == 2:
            if will_load:
                settings = get_n_distro_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", True, settings
                )
            else:
                settings = get_n_distro_generation_settings()
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", False, settings
                )
        else:
            print("Invalid classification window type given, please try again")
            done = False
    except ValueError as err:
        print("Problem found:")
        print(err)
        print("Please try again")
        done = False

experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}
experiment_neutron_data = make_strategy_for_experiments(experiment_neutron_data, factory_fn)

In [ ]:
bin_length = helpers.get_input_with_default(
    "Enter bin length (in seconds), or press Enter for default (180 s)",
    180,
    int
)
bin_string = f"{bin_length}S"

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
    'bin_length': bin_length
}

In [ ]:
analysis_timestamp

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3
overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    if fit_style == "bounds":
        # Default
        default_bounds: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
        )

        bounds_a: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
        )

        bounds_b: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
        )

        # Ranged Example
        bounds = [
            ((0, 60), bounds_a),
        ]
    else:
        default_bounds = None
        bounds = None

    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style=fit_style,
        default_bounds=default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, exp_id, nan_total_threshold=10)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    psd_report = calculate_event_time(psd_report, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

### Non-Neutron Data Processing

In [ ]:
# process reactor data files
# stored in reactor_data
# File name format: Device Param 00x
# If same device/param, but different numbers, should be merged

data_file_pattern = re.compile(r"([a-zA-Z ]+) (\d+)")
for exp_name, data_dict in experiment_neutron_data.items():
    reactor_data_folder = get_reactor_data_root(exp_name)
    time_col_name = NonReactorDataframeColumn.TIME.value
    data_col_name = NonReactorDataframeColumn.DATA.value
    units_col_name = NonReactorDataframeColumn.UNITS.value
    
    if not reactor_data_folder.is_dir():
        continue

    reactor_data_files = defaultdict(list)
    for file in reactor_data_folder.iterdir():
        match = data_file_pattern.match(file.name)
        if match and len(match.groups()) > 0:
            data_source = match.group(1)
            if isinstance(data_source, str):
                reactor_data_files[data_source].append(file)

    reactor_data = {}
    for data_source, files in reactor_data_files.items():
        file_dfs = [
            pd.read_csv(
                file,
                header=0,
                dtype=str,
                encoding='cp1252',
                names=[time_col_name, data_col_name, units_col_name]
            ) for file in sorted(files)]
        for df in file_dfs:
            df[time_col_name] = pd.to_datetime(
                df[time_col_name], utc=True)
        df = pd.concat(file_dfs, ignore_index=True)
        reactor_data[data_source] = df

    data_dict[ExperimentDataKey.REACTOR_DATA] = reactor_data

## Data Binning

In [ ]:
# Create bins
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]

    event_time_col = DetectorDataframeColumn.EVENT_TIME.value
    start_time = neutron_report[event_time_col].min()
    end_time = neutron_report[event_time_col].max()
    timetag_clock_bins = pd.date_range(
        start=start_time, end=end_time, freq=bin_string)
    data_dict[ExperimentDataKey.TIME_BIN_EDGES] = timetag_clock_bins

In [ ]:
# Bin neutron data
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    start_time = time_bins[0]

    binned_neutrons = get_time_cut(
        neutron_report, time_col_name, time_bins)
    binned_neutrons = neutron_report.groupby(
        time_bin_col_name, as_index=True) \
        .size() \
        .to_frame() \
        .copy()
    binned_neutrons.columns = [count_col_name]
    binned_neutrons[count_error_col_name] = np.sqrt(
        binned_neutrons[count_col_name]
    )

    binned_neutron_time_bins = binned_neutrons.index.to_series()
    midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
    durations = binned_neutron_time_bins.apply(
        lambda x: x.length.total_seconds()
    ).astype(np.float64)

    binned_neutrons[bin_mid_col_name] = midpoints
    binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)

    binned_neutrons[n_rate_col_name] = (
        binned_neutrons[count_col_name] / durations)
    binned_neutrons[n_error_col_name] = (
        binned_neutrons[count_error_col_name] / durations)
    binned_neutrons = binned_neutrons.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ) \
        .copy()
    data_dict[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

In [ ]:
# Bin gamma data
for exp_name, data_dict in experiment_neutron_data.items():
    gamma_report = data_dict[ExperimentDataKey.GAMMA_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    g_rate_col_name = BinningDataframeColumn.GAMMA_RATE.value
    g_error_col_name = BinningDataframeColumn.GAMMA_RATE_ERROR.value

    start_time = time_bins[0]

    binned_gamma = get_time_cut(
        gamma_report, time_col_name, time_bins)
    binned_gamma = gamma_report.groupby(
        time_bin_col_name, as_index=True) \
        .size() \
        .to_frame() \
        .copy()
    binned_gamma.columns = [count_col_name]
    binned_gamma[count_error_col_name] = np.sqrt(
        binned_gamma[count_col_name])

    binned_gamma_time_bins = binned_gamma.index.to_series()
    midpoints = binned_gamma_time_bins.apply(lambda x: x.mid)
    durations = binned_gamma_time_bins.apply(
        lambda x: x.length.total_seconds()).astype(np.float64)

    binned_gamma[bin_mid_col_name] = midpoints
    binned_gamma = bin_midpoint_time_to_seconds(binned_gamma, start_time)

    binned_gamma[g_rate_col_name] = (
        binned_gamma[count_col_name] / durations)
    binned_gamma[g_error_col_name] = (
        binned_gamma[count_error_col_name] / durations)
    binned_gamma = binned_gamma.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ).copy()
    data_dict[ExperimentDataKey.BINNED_GAMMA] = binned_gamma

In [ ]:
# bin gamma energy spectrum

def make_index_converter(
    start_time: pd.Timestamp
) -> Callable:
    def index_converter(
        category: pd.Interval
    ) -> pd.Interval:
        cat_start = category.left
        cat_end = category.right
        start_seconds = (cat_start - start_time).total_seconds()
        end_seconds = (cat_end - start_time).total_seconds()
        return pd.Interval(start_seconds, end_seconds, closed=category.closed)

    return index_converter


for exp_name, data_dict in experiment_neutron_data.items():
    gamma_report = data_dict[ExperimentDataKey.GAMMA_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
    energy_bins = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    eng_bin_col_name = BinningDataframeColumn.ENERGY_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value

    start_time = time_bins[0]

    binned_gamma_spectrum = get_time_cut(
        gamma_report, time_col_name, time_bins)
    energy_cut, energy_bins = pd.cut(
        binned_gamma_spectrum[calibrated_energy_column.value],
        bins=energy_bins,
        retbins=True
    )
    binned_gamma_spectrum[eng_bin_col_name] = energy_cut
    binned_gamma_spectrum = binned_gamma_spectrum.groupby(
        [time_bin_col_name, eng_bin_col_name],
        as_index=True
    ) \
        .size() \
        .to_frame() \
        .copy()
    binned_gamma_spectrum.columns = [count_col_name]
    binned_gamma_spectrum = binned_gamma_spectrum.reset_index(level=1)
    binned_gamma_spectrum = binned_gamma_spectrum.pivot_table(
        values=count_col_name,
        index=binned_gamma_spectrum.index,
        columns=eng_bin_col_name
    )
    
    time_index = binned_gamma_spectrum.index
    start_time = time_index[0].left
    convert_categories = make_index_converter(start_time)
    time_index = time_index.map(convert_categories)
    binned_gamma_spectrum.index = time_index

    data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM] = binned_gamma_spectrum
    data_dict[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES] = energy_bins

In [ ]:
# process and bin reactor data

def normalize_units(value, unit, to_unit):
    '''
    Converts Series of values to desired units

    value: measured value
    units: units of measured value
    to_unit: unit to convert to

    returns Series of values converted to desired unit
    '''
    try:
        return Quantity(value, unit).ito(to_unit).magnitude
    except AttributeError:
        return value


for exp_name, data_dict in experiment_neutron_data.items():
    time_col_name = NonReactorDataframeColumn.TIME.value
    data_col_name = NonReactorDataframeColumn.DATA.value
    units_col_name = NonReactorDataframeColumn.UNITS.value
    norm_data_col_name = NonReactorDataframeColumn.NORMALIZED_DATA.value
    norm_units_col_name = NonReactorDataframeColumn.NORMALIZED_UNITS.value

    if (
        reactor_data := data_dict.get(ExperimentDataKey.REACTOR_DATA)
    ) is not None:
        binned_reactor_data = {}
        time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

        for data_source, reactor_param_df in reactor_data.items():
            try:
                reactor_param_df[data_col_name] = reactor_param_df[
                    data_col_name
                ].astype(float)
            except ValueError:
                continue  # skip if not numeric

            units_counts = reactor_param_df[units_col_name].value_counts()
            main_unit = units_counts.idxmax()
            if units_counts.size > 1:
                # determine most frequent
                reactor_param_df[norm_data_col_name] = reactor_param_df.apply(
                    lambda row: normalize_units(
                        row[data_col_name],
                        row[units_col_name],
                        main_unit
                    ),
                    axis=1
                )
                reactor_param_df = reactor_param_df.assign(
                    **{norm_units_col_name: lambda _: main_unit}
                )
            else:
                reactor_param_df[norm_data_col_name] = reactor_param_df[
                    data_col_name]
                reactor_param_df[norm_units_col_name] = reactor_param_df[
                    units_col_name]

            binned_reactor_param_df = bin_non_neutron_data(
                reactor_param_df,
                time_bins,
                norm_data_col_name,
                [f"Average {data_source} ({main_unit})",
                 f"{data_source} error ({main_unit})"]
            )
            binned_reactor_data[data_source] = binned_reactor_param_df
        data_dict[ExperimentDataKey.BINNED_REACTOR_DATA] = binned_reactor_data

In [ ]:
# Merge binned data
for exp_name, data_dict in experiment_neutron_data.items():
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value

    binned_dfs = []
    binned_dfs.append(data_dict[ExperimentDataKey.BINNED_NEUTRONS])
    binned_dfs.append(data_dict[ExperimentDataKey.BINNED_GAMMA])
    binned_reactor_data = data_dict.get(ExperimentDataKey.BINNED_REACTOR_DATA)
    if binned_reactor_data is not None:
        binned_reactor_param_dfs = binned_reactor_data.values()
        for binned_reactor_param_df in binned_reactor_param_dfs:
            binned_dfs.append(binned_reactor_param_df)
    merged_df = reduce(
        lambda df1, df2: pd.merge(
            df1, df2, how='left', on=[
                time_bin_col_name, bin_mid_col_name, bin_time_col_name
            ]
        ),
        binned_dfs
    )
    data_dict[ExperimentDataKey.ALL_BINNED_DATA] = merged_df

### Define Voltage Regions

In [ ]:
# list(experiment_neutron_data["TB-15"].keys())
experiment_neutron_data["TB-15"][ExperimentDataKey.ALL_BINNED_DATA]

In [ ]:
tb_15_voltage_regions = [
    (12,66,30),
    (69,81,20),
    (84,96,15),
    (99,111,25),
    (114,126,20)
]
tb_16_voltage_regions = [
    (10,66,25),
    (69,82,20),
    (85,96,15),
    (99,174,30),
    (177,188,25),
    (191,204,20),
    (207,222,15)
]
tb_17_voltage_regions = [
    (12,82,25),
    (85,97,20),
    (100,112,15),
    (115,172,30),
    (175,187,25),
    (190,202,20),
    (205,217,15)
]
keys = ["TB-15", "TB-16", "TB-17"]
voltage_regions = {k: v for k, v in zip(keys, [tb_15_voltage_regions, tb_16_voltage_regions, tb_17_voltage_regions])}

In [ ]:
voltage_rate_data = {"Voltage (kV)": [], "Average rate (cps)": [], "Rate error (cps)": []}
for exp_id, voltage_region_list in voltage_regions.items():
    data_dict = experiment_neutron_data[exp_id]
    binned_data = data_dict[ExperimentDataKey.ALL_BINNED_DATA]
    for voltage_region in voltage_region_list:
        region_start, region_end, voltage = voltage_region
        region_start = region_start * 60
        region_end = region_end * 60
        region_data = binned_data.query("`Bin time (s)`.between(@region_start, @region_end)")
        avg_rate = region_data['Neutron rate (cps)'].mean()
        rate_error = max(region_data['Neutron rate (cps)'].std(), region_data['Neutron error (cps)'].max())
        voltage_rate_data["Voltage (kV)"].append(voltage)
        voltage_rate_data["Average rate (cps)"].append(avg_rate)
        voltage_rate_data["Rate error (cps)"].append(rate_error)

In [ ]:
voltages = voltage_rate_data["Voltage (kV)"]
rates = voltage_rate_data["Average rate (cps)"]
rate_errors = voltage_rate_data["Rate error (cps)"]

def exponential(x, a, b):
    return a * np.exp(b*x)

params, cov = curve_fit(exponential, voltages, rates, sigma=rate_errors)
print(params)
print(np.sqrt(np.diag(cov)))

In [ ]:
figsize = (12, 10)

voltages = voltage_rate_data["Voltage (kV)"]
rates = voltage_rate_data["Average rate (cps)"]
rate_errors = voltage_rate_data["Rate error (cps)"]

a, b = params
x = np.linspace(15,30,100)
y = exponential(x, a, b)

fig, ax = plt.subplots(figsize=figsize)
ax.errorbar(voltages, rates, yerr=rate_errors, fmt="b.", linestyle="")
ax.plot(x, y, color="black", linestyle="--")
annot_x = 28
annot_y = exponential(annot_x, a, b)
ax.annotate(fr"$f(x)={a:.2f}e^{{{b:.2f}x}}$", (annot_x, annot_y), (annot_x - 3, annot_y + 10), arrowprops={"width": 1, "headwidth": 4}, bbox={"boxstyle": "round", "facecolor": "white"})
ax.set(xlabel="Voltage (kV)", ylabel="Average rate (cps)", yscale="log")
plt.show()